In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 🔎 Étape 4 : Analyse Exploratoire des Données (EDA)

Cette étape correspond au quatrième chapitre du cours. L'objectif est d'explorer et de résumer les propriétés statistiques fondamentales de vos données et de réaliser du **Feature Engineering** pour enrichir vos modèles.

### 1. Préparation de l'environnement

In [2]:
import os
import sys
import pandas as pd
import numpy as np

print("Librairies importées pour l'EDA !")

Librairies importées pour l'EDA !


### 2. Chargement des données nettoyées

In [3]:
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
df.head()

,Date,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-01-01 00:19:34,"""CNR4352144""",Completed,"""CID8362794""",Bike,Udyog Vihar,Ambience Mall,0.007125,28.343157,NaN,NaN,NaN,NaN,NaN,NaN,419.656808,54.403511,4.8,4.8,Cash
1,2024-01-01 01:35:18,"""CNR9147645""",Completed,"""CID8300238""",Go Mini,Basai Dhankot,Madipur,8.051596,21.722728,NaN,NaN,NaN,NaN,NaN,NaN,272.677149,40.481723,4.2,4.1,Uber Wallet
2,2024-01-01 01:37:50,"""CNR1009222""",Cancelled by Driver,"""CID2030746""",Go Sedan,Tughlakabad,Greater Kailash,0.000000,0.000000,NaN,NaN,1.0,More than permitted people in there,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN
3,2024-01-01 01:48:03,"""CNR2740479""",Cancelled by Driver,"""CID3231181""",Auto,Palam Vihar,Kherki Daula Toll,0.000000,0.000000,NaN,NaN,1.0,Personal & Car related issues,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN
4,2024-01-01 01:49:56,"""CNR7650148""",Cancelled by Driver,"""CID3381661""",Go Sedan,Narsinghpur,Pulbangash,0.000000,0.000000,NaN,NaN,1.0,More than permitted people in there,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN


### 3. Statistiques Descriptives

On peut générer des résumés statistiques globaux et par groupes/catégories de notre jeu de données via la fonction describe().

In [4]:
df.describe()

,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Cancelled Rides by Driver,Incomplete Rides,Booking Value,Ride Distance,Driver Ratings,Customer Rating
count,150000.000000,150000.000000,10500.0,27000.0,9000.0,150000.000000,150000.000000,93000.000000,93000.000000
mean,9.655084,18.320726,1.0,1.0,1.0,589.426500,28.250391,4.230992,4.404584
std,10.784205,16.473676,0.0,0.0,0.0,590.612943,21.172459,0.436871,0.437819
min,0.000000,0.000000,1.0,1.0,1.0,0.000000,0.000000,3.000000,3.000000
25%,0.000000,0.000000,1.0,1.0,1.0,0.000000,0.000000,4.100000,4.200000
50%,8.695344,17.671802,1.0,1.0,1.0,498.519855,33.961385,4.300000,4.500000
75%,14.986856,30.133560,1.0,1.0,1.0,921.927285,44.883307,4.600000,4.800000
max,123.192593,96.126316,1.0,1.0,1.0,5451.396529,79.060608,5.000000,5.000000


### 4. Triage des colonnes pertinentes
On va chercher à sélectionner les colonnes pertinentes pour notre étude. On triera parmi :
- La colonne sur laquelle nous voudrons faire notre étude : Booking Value
- Les colonnes qui sont des variables susceptibles d'influer sur le prix de la course (Date, Vehicle Type, Pickup Location, Drop Location, Avg VTAT, Avg CTAT, Ride Distance, Payment Method – cette dernière est contre-intuitive, mais on peut supposer des tarifs variant selon le moyen de paiement). Nous garderons toutes ces colonnes, en les reformatant éventuellement.
- Les colonnes qui n'ont aucune valeur dans notre étude (Booking ID, Customer ID, Driver Ratings, Customer Rating) ; les deux premières n'ont aucune raison d'avoir un impact sur le prix de la course (si l'on exclut l'idée que le chauffeur puisse faire payer à la tête), les deux suivantes peuvent être corrélées mais sont des données obtenues après la détermination du prix de la course, donc n'auront aucun effet causal. Nous supprimerons toutes ces colonnes.
- Les colonnes qui traitent du statut de la course (Booking Status, Cancelled Rides by Customer, Reason for cancelling by Customer, Cancelled Rides by Driver, Driver Cancellation Reason, Incomplete Rides, Incomplete Rides Reason). Nous traiterons ces colonnes à part.

#### Traitement des colonnes de statut

En faisant un value_counts() sur Booking Status, nous pouvons constater que cette colonne a cinq valeurs possibles ("Completed", "Cancelled by Driver", "No Driver Found", "Cancelled by Customer", "Incomplete"), qui correspondent exactement aux valeurs non-nulles sur les colonnes correspondant à ces issues possibles. Par ailleurs, en étudiant les valeurs minimales et maximales du montant de la course en filtrant par Booking Status, on peut vérifier que les courses ayant le statut "Cancelled by Driver", "No Driver Found" et "Cancelled by Customer" ont toutes des valeurs nulles sur Booking Value (la course étant interrompue, il n'y a pas de paiement), tandis que celles ayant le statut Completed ou Incomplete en ont une.

Nous supprimerons donc tous les enregistrements ayant des statuts éliminatoires, ainsi que toutes les colonnes correspondant à ces statuts qui ne serviront donc plus à rien. Au terme de cette opération, nous pouvons constater qu'il n'y a plus aucune valeur nulle, donc nous n'aurons plus à nous préoccuper de cette question.

In [5]:
df["Booking Status"].value_counts()

Booking Status
Completed                93000
Cancelled by Driver      27000
No Driver Found          10500
Cancelled by Customer    10500
Incomplete                9000
Name: count, dtype: int64

In [6]:
print("Statuts possibles :\n", df["Booking Status"].value_counts())

# Vérifier s'il y a des paiements en fonction des statuts.
print(f"""\nPrix minimum en fonction du statut :
      Completed : {df[df["Booking Status"] == "Completed"]["Booking Value"].min()}
      Incomplete : {df[df["Booking Status"] == "Incomplete"]["Booking Value"].min()}
      No Driver Found : {df[df["Booking Status"] == "No Driver Found"]["Booking Value"].min()}
      Cancelled by Driver : {df[df["Booking Status"] == "Cancelled by Driver"]["Booking Value"].min()}
      Cancelled by Customer : {df[df["Booking Status"] == "Cancelled by Customer"]["Booking Value"].min()}""")

# On constate que seules les courses "Completed" et "Incomplete" ont des paiements

Statuts possibles :
 Booking Status
Completed                93000
Cancelled by Driver      27000
No Driver Found          10500
Cancelled by Customer    10500
Incomplete                9000
Name: count, dtype: int64

Prix minimum en fonction du statut :
      Completed : 31.81391514303704
      Incomplete : 46.877889667812966
      No Driver Found : 0.0
      Cancelled by Driver : 0.0
      Cancelled by Customer : 0.0


In [7]:
# On retire les colonnes qui ne seront pas pertinentes et les lignes n'impliquant aucun paiement
to_drop = ['Customer ID', 'Booking ID', 'Driver Ratings', 'Customer Rating', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason']
df = df[(df["Booking Status"] == "Completed") | (df["Booking Status"] == "Incomplete")]
df = df.drop(to_drop, axis=1)

In [8]:
# On n'a plus aucun NaN dans la base (ils étaient tous dans les colonnes qu'on a supprimées)
df.isna().sum()

Date               0
Booking Status     0
Vehicle Type       0
Pickup Location    0
Drop Location      0
Avg VTAT           0
Avg CTAT           0
Booking Value      0
Ride Distance      0
Payment Method     0
dtype: int64

### 5. Encodage des colonnes

On va maintenant encoder les colonnes à valeurs nominales :
- La colonne "Booking Status" n'ayant plus que deux valeurs possibles (Completed et Incomplete), on peut en faire une colonne booléenne "Completed" aux valeurs True et False.
- Les colonnes "Vehicle Type" et "Payment Method" ont respectivement 7 et 5 valeurs possibles – on le vérifie avec la fonction unique() ; on peut envisager de l'encoder en OHE, mais on se contentera d'un target encoding.
- Les colonnes "Pickup Location" et "Drop Location" ont 176 valeurs possibles ; on préférera le target encoding dans ce cas.

In [9]:
print(f"""# Valeurs possibles :
      Vehicle Type : {len(df["Vehicle Type"].unique())}
      Pickup Location : {len(df["Pickup Location"].unique())}
      Drop Location : {len(df["Drop Location"].unique())}
      Payment Method : {len(df["Payment Method"].unique())}""")

# Valeurs possibles :
      Vehicle Type : 7
      Pickup Location : 176
      Drop Location : 176
      Payment Method : 5


In [10]:
status_encoder = {"Completed":True, "Incomplete":False}
df["Completed"] = df["Booking Status"].replace(status_encoder)
df = df.drop("Booking Status", axis=1)

vehicle_encoder = {}
for i, label in enumerate(df["Vehicle Type"].unique()) :
    vehicle_encoder[label] = i
df["Vehicle Type"] = df["Vehicle Type"].replace(vehicle_encoder)

payment_encoder = {}
for i, label in enumerate(df["Payment Method"].unique()) :
    payment_encoder[label] = i
df["Payment Method"] = df["Payment Method"].replace(payment_encoder)

location_encoder = {}
for i, label in enumerate(df["Pickup Location"].unique()) :
    location_encoder[label] = i
df["Pickup Location"] = df["Pickup Location"].replace(location_encoder)
df["Drop Location"] = df["Drop Location"].replace(location_encoder)

C:\Users\utilisateur\AppData\Local\Temp\ipykernel_15192\1822198833.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Completed"] = df["Booking Status"].replace(status_encoder)
C:\Users\utilisateur\AppData\Local\Temp\ipykernel_15192\1822198833.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Vehicle Type"] = df["Vehicle Type"].replace(vehicle_encoder)
C:\Users\utilisateur\AppData\Local\Temp\ipykernel_15192\1822198833.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future v

### 6. Sauvegarde des données traitées

On enregistre nos données traitées dans le répertoire `data/processed/`.

In [11]:
# Aperçu du dataset final :
df

,Date,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Booking Value,Ride Distance,Payment Method,Completed
0,2024-01-01 00:19:34,0,0,18,0.007125,28.343157,419.656808,54.403511,0,True
1,2024-01-01 01:35:18,1,1,30,8.051596,21.722728,272.677149,40.481723,1,True
5,2024-01-01 01:53:01,1,2,114,6.290125,37.409325,554.320877,52.917495,2,True
13,2024-01-01 03:59:29,1,3,30,12.022176,25.064132,276.015798,33.224052,0,True
14,2024-01-01 04:00:07,2,4,116,21.502254,10.936425,187.665321,36.051597,0,True
...,...,...,...,...,...,...,...,...,...,...
149995,2024-12-30 22:58:00,0,169,42,14.969327,11.071073,476.640730,60.222935,2,True
149996,2024-12-30 23:03:14,2,11,88,12.493995,34.215069,775.914923,40.883461,2,True
149997,2024-12-30 23:17:05,1,162,49,2.113810,55.446784,689.833540,24.913603,2,True
149998,2024-12-30 23:21:12,5,28,14,5.638634,29.682250,799.738284,50.490099,1,True


In [12]:
processed_path = '../data/processed/cleaned_data_sample.csv'
# Sauvegarde avec df.to_csv()
df.to_csv(processed_path, index=False)
print(f"💾 Données propres sauvegardées dans : {processed_path}")

💾 Données propres sauvegardées dans : ../data/processed/cleaned_data_sample.csv
